# AACAgent — Metric Evaluation

Loads one or two CSV files produced by `eval.ipynb` (Colab sample + cluster full dataset) and computes all evaluation metrics.

**Input:** `eval/results/eval_hf.csv` (cluster) and/or `eval/results/eval_colab.csv` (Colab sample).  

## 0 · Config — set your CSV paths here

In [ ]:
from pathlib import Path
import sys

# ── Project root ──────────────────────────────────────────────────────────────
_NB_DIR = Path().resolve()
if (_NB_DIR / 'eval' / 'metric_evaluation.ipynb').exists():
    PROJECT_ROOT = _NB_DIR
else:
    PROJECT_ROOT = _NB_DIR.parent

RESULTS_DIR = PROJECT_ROOT / 'eval' / 'results'

# ── CSV paths — edit if your filenames differ ────────────────────────────────
# Set to None to skip a source.
CSV_CLUSTER = RESULTS_DIR / 'eval_hf.csv'        # full-dataset run from cluster
CSV_COLAB   = RESULTS_DIR / 'eval_colab.csv'     # sampled run from Colab (optional)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Cluster CSV  : {CSV_CLUSTER}  exists={CSV_CLUSTER.exists()}')
print(f'Colab CSV    : {CSV_COLAB}   exists={CSV_COLAB.exists()}')

## 1 · Imports

In [ ]:
import ast
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Imports OK.')

## 2 · Load and merge CSVs

In [ ]:
def load_csv(path: Path, source_label: str) -> Optional[pd.DataFrame]:
    """Load an eval CSV and add a `source` column."""
    if path is None or not path.exists():
        print(f'  [SKIP] {path} — not found.')
        return None
    df = pd.read_csv(path)
    # Parse list columns
    for col in ('predicted_ids', 'all_gold_ids', 'resolve_queries'):
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
    df['source'] = source_label
    print(f'  Loaded {len(df):,} rows from {path.name}  ({source_label})')
    return df


frames = []
df_cluster = load_csv(CSV_CLUSTER, 'cluster')
df_colab   = load_csv(CSV_COLAB,   'colab')
if df_cluster is not None: frames.append(df_cluster)
if df_colab   is not None: frames.append(df_colab)

assert frames, 'No CSV files found — please check the paths in cell 0.'
res = pd.concat(frames, ignore_index=True)

print(f'\nCombined rows : {len(res):,}')
print(f'Models        : {sorted(res["model"].unique())}')
print(f'Splits        : {sorted(res["split"].unique())}')
print(f'Sources       : {sorted(res["source"].unique())}')
res.head(3)

## 3 · Overview

In [ ]:
print('=== Dataset overview ===')
print(res.groupby(['model', 'split', 'source'])[['hit', 'gold_in_candidates']].agg(
    n_turns=('hit', 'count'),
    n_rows=('hit', lambda x: res.loc[x.index, 'row_idx'].nunique()),
    hit_rate=('hit', 'mean'),
    retrieval_rate=('gold_in_candidates', 'mean'),
).round(3).to_string())

## 4 · Hit@Window — primary metric

In [ ]:
OVERLAP_LEVELS = ['synset', 'category', 'keyword', 'tag']

def print_hit_at_window(df: pd.DataFrame) -> pd.DataFrame:
    """Compute hit@window overall, first-turn, and late-turn."""
    rows = []
    for (model, split), g in df.groupby(['model', 'split']):
        first   = g[g['turn_pos'] == 0]
        late    = g[g['turn_pos'] >  0]
        rows.append({
            'model':          model,
            'split':          split,
            'n_turns':        len(g),
            'n_rows':         g['row_idx'].nunique(),
            'hit_all':        g['hit'].mean(),
            'hit_turn0':      first['hit'].mean() if len(first) else float('nan'),
            'hit_turnN':      late['hit'].mean()  if len(late)  else float('nan'),
            'retrieval_rate': g['gold_in_candidates'].mean(),
        })
    tbl = pd.DataFrame(rows).set_index(['model', 'split'])
    print(tbl.round(3).to_string())
    return tbl


hit_tbl = print_hit_at_window(res)

In [ ]:
# ── Bar chart — hit@window by model × split ───────────────────────────────────
pivot = hit_tbl.reset_index().pivot(index='model', columns='split', values='hit_all')
ax = pivot.plot(kind='bar', figsize=(8, 4), edgecolor='white', width=0.6)
ax.set_title('Hit@Window by model and split')
ax.set_ylabel('Hit rate')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend(title='split')
plt.tight_layout()
plt.show()

## 5 · Failure breakdown

In [ ]:
def failure_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    """Decompose failures into retrieval failures vs LLM ranking failures."""
    rows = []
    for (model, split), g in df.groupby(['model', 'split']):
        n              = len(g)
        retrieval_fail = (~g['gold_in_candidates']).sum()
        llm_fail       = (~g[g['gold_in_candidates']]['hit']).sum()
        success        = g['hit'].sum()
        rows.append({
            'model':              model,
            'split':              split,
            'n':                  n,
            'retrieval_fail':     retrieval_fail,
            'retrieval_fail_pct': retrieval_fail / n,
            'llm_fail':           llm_fail,
            'llm_fail_pct':       llm_fail / n,
            'success':            success,
            'success_pct':        success / n,
        })
    tbl = pd.DataFrame(rows).set_index(['model', 'split'])
    print(tbl.round(3).to_string())
    return tbl


fail_tbl = failure_breakdown(res)

In [ ]:
# Stacked bar: retrieval_fail / llm_fail / success per model×split
labels = [f'{m}\n{s}' for m, s in fail_tbl.index]
fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.4), 4))
bottom = np.zeros(len(labels))
for col, color, label in [
    ('retrieval_fail_pct', '#e57373', 'Retrieval fail'),
    ('llm_fail_pct',       '#ffb74d', 'LLM ranking fail'),
    ('success_pct',        '#81c784', 'Success (hit)'),
]:
    vals = fail_tbl[col].values
    ax.bar(labels, vals, bottom=bottom, color=color, label=label, edgecolor='white')
    bottom += vals
ax.set_title('Failure breakdown by model and split')
ax.set_ylabel('Fraction of turns')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 6 · Semantic overlap levels

In [ ]:
def semantic_overlap_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, split), g in df.groupby(['model', 'split']):
        row = {'model': model, 'split': split,
               'semantic_hit': g['overlap_level'].notna().mean()}
        for lvl in OVERLAP_LEVELS:
            row[lvl] = (g['overlap_level'] == lvl).mean()
        row['none'] = g['overlap_level'].isna().mean()
        rows.append(row)
    tbl = pd.DataFrame(rows).set_index(['model', 'split'])
    print(tbl.round(3).to_string())
    return tbl


overlap_tbl = semantic_overlap_table(res)

In [ ]:
# Stacked bar: overlap levels
cols   = OVERLAP_LEVELS + ['none']
colors = ['#4fc3f7', '#aed581', '#fff176', '#ffcc80', '#e0e0e0']
labels = [f'{m}\n{s}' for m, s in overlap_tbl.index]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.4), 4))
bottom = np.zeros(len(labels))
for col, color in zip(cols, colors):
    vals = overlap_tbl[col].values
    ax.bar(labels, vals, bottom=bottom, color=color, label=col, edgecolor='white')
    bottom += vals
ax.set_title('Semantic overlap level distribution')
ax.set_ylabel('Fraction of turns')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 7 · Resolve method distribution

In [ ]:
RESOLVE_METHODS = ['exact', 'lemma', 'hyphen', 'lemma_alt', 'token', 'none']

def resolve_method_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, split), g in df.groupby(['model', 'split']):
        row = {'model': model, 'split': split}
        for method in RESOLVE_METHODS:
            mask = g['resolve_method'] == method
            row[f'{method}_pct']      = mask.mean()
            row[f'{method}_hit_rate'] = g.loc[mask, 'hit'].mean() if mask.any() else float('nan')
        rows.append(row)
    tbl = pd.DataFrame(rows).set_index(['model', 'split'])
    # Print only pct columns for readability
    pct_cols = [f'{m}_pct' for m in RESOLVE_METHODS]
    print('Resolve method distribution (pct):')
    print(tbl[pct_cols].round(3).to_string())
    print('\nHit rate per resolve method:')
    hit_cols = [f'{m}_hit_rate' for m in RESOLVE_METHODS]
    print(tbl[hit_cols].round(3).to_string())
    return tbl


resolve_tbl = resolve_method_table(res)

## 8 · Plan method distribution

In [ ]:
PLAN_METHODS = ['llm', 'fallback_spacy', 'fallback_empty']

def plan_method_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, split), g in df.groupby(['model', 'split']):
        row = {'model': model, 'split': split}
        for method in PLAN_METHODS:
            mask = g['plan_method'] == method
            row[f'{method}_pct']      = mask.mean()
            row[f'{method}_hit_rate'] = g.loc[mask, 'hit'].mean() if mask.any() else float('nan')
        rows.append(row)
    tbl = pd.DataFrame(rows).set_index(['model', 'split'])
    pct_cols = [f'{m}_pct' for m in PLAN_METHODS]
    print('Plan method distribution (pct):')
    print(tbl[pct_cols].round(3).to_string())
    print('\nHit rate per plan method:')
    hit_cols = [f'{m}_hit_rate' for m in PLAN_METHODS]
    print(tbl[hit_cols].round(3).to_string())
    return tbl


plan_tbl = plan_method_table(res)

## 9 · Synset expansion and rank source

In [ ]:
def synset_expansion_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, split), g in df.groupby(['model', 'split']):
        rows.append({
            'model':                model,
            'split':                split,
            'expanded_turns_pct':   (g['synset_added'] > 0).mean(),
            'avg_synset_added':     g['synset_added'].mean(),
            'window_fully_fresh':   (g['fresh_count'] == g['window_len']).mean(),
            'window_padded_pct':    (g['fresh_count'] < g['window_len']).mean(),
            'avg_fresh_count':      g['fresh_count'].mean(),
        })
    tbl = pd.DataFrame(rows).set_index(['model', 'split'])
    print(tbl.round(3).to_string())
    return tbl


synset_tbl = synset_expansion_table(res)

## 10 · Tool-call behaviour

Expected: `clear` split → 0% tool calls; `vague` split → ~100% tool calls.

In [ ]:
def tool_call_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, split), g in df.groupby(['model', 'split']):
        rows.append({
            'model':            model,
            'split':            split,
            'get_time_pct':     g['called_get_time'].mean(),
            'get_schedule_pct': g['called_get_schedule'].mean(),
        })
    tbl = pd.DataFrame(rows).set_index(['model', 'split'])
    print(tbl.round(3).to_string())
    return tbl


tool_tbl = tool_call_table(res)

## 11 · Turn-position analysis

Does performance degrade for later turns in a multi-turn sequence?

In [ ]:
models = sorted(res['model'].unique())
splits = sorted(res['split'].unique())
n_plots = len(models) * len(splits)

fig, axes = plt.subplots(
    len(models), len(splits),
    figsize=(5 * len(splits), 3.5 * len(models)),
    squeeze=False,
)
for r, model in enumerate(models):
    for c, split in enumerate(splits):
        ax   = axes[r][c]
        g    = res[(res['model'] == model) & (res['split'] == split)]
        pos  = g.groupby('turn_pos')['hit'].mean()
        ax.bar(pos.index, pos.values, color='#5c9eed', edgecolor='white')
        ax.set_title(f'{model.split("/")[-1]}\n{split}')
        ax.set_xlabel('Turn position')
        ax.set_ylabel('Hit rate')
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
        ax.set_xticks(pos.index)

plt.suptitle('Hit rate by turn position', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 12 · Model comparison summary

In [ ]:
summary = res.groupby(['model', 'split']).agg(
    n_turns            = ('hit',                'count'),
    hit_rate           = ('hit',                'mean'),
    retrieval_rate     = ('gold_in_candidates', 'mean'),
    semantic_hit_rate  = ('overlap_level',      lambda x: x.notna().mean()),
    synset_exp_pct     = ('synset_added',        lambda x: (x > 0).mean()),
    get_time_pct       = ('called_get_time',     'mean'),
    get_schedule_pct   = ('called_get_schedule', 'mean'),
).round(3)

print('=== Full summary table ===')
print(summary.to_string())
summary

## 13 · Cluster vs Colab comparison (if both CSVs are available)

In [ ]:
if res['source'].nunique() > 1:
    comp = res.groupby(['model', 'split', 'source'])[['hit', 'gold_in_candidates']].agg(
        n_turns=('hit', 'count'),
        hit_rate=('hit', 'mean'),
        retrieval_rate=('gold_in_candidates', 'mean'),
    ).round(3)
    print('=== Cluster vs Colab comparison ===')
    print(comp.to_string())
else:
    print('Only one source loaded — cross-source comparison not available.')